# Maia Lite 0.3.0 — Auditoria contextual e curadoria pré-SFT v2.4

**Objetivo:** cruzar o corpus SFT v2.2/v2.3 com a auditoria qualitativa e o benchmark, priorizando revisão de contexto ausente, precisão factual e redundância. **Não modifica os dados de origem, não remove exemplos automaticamente e não treina modelos.**

A ligação entre exemplos do corpus e falhas de geração é **exploratória**: similaridade temática não demonstra causalidade. Heurísticas indicam itens para revisão, não erros comprovados.

Execução: Colab CPU ou Jupyter local. Configure `ROOT` e `INPUT_DIR` na célula abaixo. Aceita os CSVs anexados nesta conversa ou arquivos no Drive. Os arquivos do benchmark são opcionais.

In [ ]:
from pathlib import Path
import json, re, math, hashlib, unicodedata
import pandas as pd
import numpy as np
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
INPUT_DIR = ROOT / 'audits' / 'sft_v2_2_v2_3_qualitative_v1'
DATASET_DIR = ROOT / 'datasets' / 'sft_v2_2_aya'
OUTPUT_DIR = ROOT / 'audits' / 'sft_v2_2_v2_3_contextual_v2'
# Alternativa local: ROOT=Path('/mnt/data'); INPUT_DIR=ROOT; DATASET_DIR=ROOT; OUTPUT_DIR=ROOT/'contextual_audit_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED=20261120
MAX_REVIEW=420
print('Saída:',OUTPUT_DIR)


## 1. Localização segura dos arquivos

O notebook procura somente nas pastas configuradas e em subpastas próximas. Se não localizar a amostra, interrompe com mensagem clara; não substitui por dados inventados.

In [ ]:
def locate(names, roots, required=False):
    for root in roots:
        root=Path(root)
        for name in names:
            p=root/name
            if p.is_file(): return p
        if root.is_dir():
            for name in names:
                hits=sorted(root.glob('**/'+name))
                if hits:return hits[0]
    if required: raise FileNotFoundError(f'Não encontrado: {names}. Ajuste INPUT_DIR / DATASET_DIR. Pastas: {roots}')
    return None

roots=[INPUT_DIR, DATASET_DIR, ROOT, Path('/mnt/data'), Path('/content')]
review_path=locate(['human_review_corpus.csv'], roots, required=True)
risk_path=locate(['risk_signals.csv'], roots)
benchmark_path=locate(['paired_responses(1).csv','paired_responses.csv'], roots)
generations_path=locate(['generations(1).csv','generations.csv'], roots)
train_path=locate(['train.jsonl','sft_train.jsonl','train-pairs.jsonl'],[DATASET_DIR,ROOT,Path('/mnt/data')])
val_path=locate(['validation.jsonl','sft_validation.jsonl'],[DATASET_DIR,ROOT,Path('/mnt/data')])
for name,p in [('review',review_path),('risk',risk_path),('benchmark',benchmark_path),('generations',generations_path),('train',train_path),('validation',val_path)]:print(name,':',p)


## 2. Leitura, normalização e preservação de avaliações existentes

In [ ]:
review=pd.read_csv(review_path, keep_default_na=False, dtype=str).fillna('')
required={'review_id','language','prompt','completion','source','split','id'}
missing=required-set(review.columns)
if missing:raise ValueError(f'Colunas ausentes na revisão: {sorted(missing)}')
if review['review_id'].duplicated().any():raise ValueError('review_id duplicado')
risk=pd.read_csv(risk_path, dtype=str).fillna('') if risk_path else pd.DataFrame()
if not risk.empty and 'review_id' in risk.columns:
    risk=risk.drop_duplicates('review_id')
    review=review.merge(risk[['review_id']+[c for c in ['risk_score','repeated_trigrams','very_short','long_answer','numeric_not_in_prompt','unqualified_absolute'] if c in risk]],on='review_id',how='left',suffixes=('','_risk'))
print('Registros de revisão:',len(review));print(review.groupby(['language','source']).size().to_string())


## 3. Detectores conservadores de contexto ausente

Sinais multilíngues: referências anafóricas, pedidos de continuação, agradecimentos sem pergunta independente, mudança de algo anterior. São pistas; exemplos devem ser lidos por uma pessoa.

In [ ]:
CONTEXT_PATTERNS={
 'followup':r'(?i)^\s*(?:now|then|also|and|yes|thanks|thank you|great|ok(?:ay)?|actually|again|continue|please continue|ahora|entonces|también|sí|gracias|continúa|sigue|agora|então|também|sim|obrigad[oa]|continue|prossiga)\b',
 'change_previous':r'(?i)\b(?:change|modify|rewrite|revise|adjust|edit|replace|alter|cambia|modifica|reescribe|altera|mude|altere|reescreva|modifique)\b.{0,65}\b(?:it|that|this|the previous|the above|lo|eso|esto|anterior|isso|isto|aquilo|anterior)\b',
 'reference_previous':r'(?i)\b(?:as (?:you|we) (?:said|mentioned)|earlier|previous (?:answer|message|example)|above|same (?:as|thing)|como dijiste|antes mencionado|lo anterior|como você disse|mensagem anterior|resposta anterior|acima)\b',
 'deictic_short':r'(?i)^\s*(?:do (?:it|that)|make (?:it|that)|tell me more|what about (?:that|it)|hazlo|haz eso|dime más|faça isso|faça o mesmo|e isso|e aquele)\s*[?.!]*\s*$'
}
for key,pattern in CONTEXT_PATTERNS.items():review['ctx_'+key]=review['prompt'].str.contains(pattern,regex=True,na=False).astype(int)
review['context_risk']=review[['ctx_'+k for k in CONTEXT_PATTERNS]].sum(axis=1)
review['context_risk_label']=np.select([review.context_risk>=2,review.context_risk==1],['high','review'],default='none')
print(review.groupby(['source','context_risk_label']).size().unstack(fill_value=0).to_string())


## 4. Sinais de qualidade da resposta

Não confundir números não presentes no prompt com alucinação. A verificação factual exige fonte ou especialista.

In [ ]:
def repeat_rate(t):
    tokens=re.findall(r'\w+',str(t).lower())
    tris=list(zip(tokens,tokens[1:],tokens[2:]))
    return 0.0 if not tris else (len(tris)-len(set(tris)))/len(tris)

def echo_ratio(p,a):
    x=set(re.findall(r'\w+',str(p).lower()));y=set(re.findall(r'\w+',str(a).lower()))
    return len(x&y)/max(1,len(y))

review['answer_words']=review['completion'].map(lambda x:len(str(x).split()))
review['repeat_rate']=review['completion'].map(repeat_rate)
review['echo_ratio']=review.apply(lambda r:echo_ratio(r['prompt'],r['completion']),axis=1)
review['quality_repetition_flag']=(review['repeat_rate']>=0.18).astype(int)
review['quality_very_short_flag']=(review['answer_words']<=8).astype(int)
review['quality_very_long_flag']=(review['answer_words']>=220).astype(int)
review['quality_echo_flag']=((review['echo_ratio']>=0.9)&(review['answer_words']>=12)).astype(int)
review['quality_risk']=review[['quality_repetition_flag','quality_very_short_flag','quality_very_long_flag','quality_echo_flag']].sum(axis=1)
print(review[['answer_words','repeat_rate','echo_ratio']].describe().to_string())


## 5. Cruzamento exploratório com o benchmark

Apenas vincula por similaridade lexical **dentro do mesmo idioma**. Exige pelo menos 2 palavras informativas compartilhadas. O vínculo é um indício para revisão, **não** a origem demonstrada do erro.

In [ ]:
STOP={
 'pt':set('de da do dos das o a os as e em um uma para por que qual quais como é são foi se não com na no ao aos as sobre explique responda forma clara objetiva cientificamente correta'.split()),
 'en':set('the a an and or of in on to for what why how is are was were do does did if not can you your answer clearly concisely accurately with from about'.split()),
 'es':set('de del la el los las un una y en para por que cuál cuáles cómo es son fue si no con al sobre responde forma clara precisa'.split())}
def tokens(t,lang):
    t=unicodedata.normalize('NFKD',str(t)).encode('ascii','ignore').decode().lower()
    return {w for w in re.findall(r'[a-z]{3,}',t) if w not in STOP.get(lang,set())}

pairs=pd.DataFrame()
if benchmark_path:
    candidate=pd.read_csv(benchmark_path).fillna('')
    if {'question_id','language','response_v2.2','response_v2.3','prompt'}.issubset(candidate.columns):pairs=candidate
if pairs.empty and generations_path:
    gen=pd.read_csv(generations_path).fillna('')
    if {'question_id','model','prompt','language','response','format','strategy'}.issubset(gen.columns):
        pairs=gen.pivot_table(index=['question_id','language','prompt','format','strategy'],columns='model',values='response',aggfunc='first').reset_index()
        pairs.columns.name=None
        pairs=pairs.rename(columns={'v2.2':'response_v2.2','v2.3':'response_v2.3'})

links=[]
if not pairs.empty:
    pairs=pairs.copy()
    pairs['tokens']=pairs.apply(lambda r:tokens(r['prompt'],r['language']),axis=1)
    for row in review.itertuples(index=False):
        rt=tokens(row.prompt,row.language)
        if len(rt)<2:continue
        candidates=pairs[pairs.language==row.language]
        scores=[]
        for j,bm in candidates.iterrows():
            overlap=len(rt&bm['tokens'])
            score=overlap/max(1,len(rt|bm['tokens']))
            if overlap>=2 and score>=0.10:scores.append((score,overlap,j))
        if scores:
            score,overlap,j=max(scores)
            bm=pairs.loc[j]
            links.append({'review_id':row.review_id,'corpus_id':row.id,'language':row.language,'benchmark_question_id':bm['question_id'],'benchmark_format':bm.get('format',''),'benchmark_strategy':bm.get('strategy',''),'lexical_jaccard':round(score,4),'overlap_words':overlap,'benchmark_prompt':bm['prompt'],'benchmark_v2_2':bm.get('response_v2.2',''),'benchmark_v2_3':bm.get('response_v2.3',''),'link_status':'exploratory_not_causal'})
links_df=pd.DataFrame(links)
print('Benchmark disponível:',not pairs.empty,'| ligações exploratórias:',len(links_df))


## 6. Priorização transparente de revisão

Não exclui registros. Prioridade alta quando há múltiplos indícios de contexto ou repetição; o julgamento final permanece humano.

In [ ]:
review['priority_score']=(review['context_risk']*3+review['quality_risk']*2).astype(int)
if not links_df.empty:
    counts=links_df.groupby('review_id').size().rename('benchmark_links')
    review=review.join(counts,on='review_id')
else:review['benchmark_links']=0
review['benchmark_links']=review['benchmark_links'].fillna(0).astype(int)
review['priority_score']+=np.minimum(review['benchmark_links'],2)
review['priority']=np.select([review.priority_score>=5,review.priority_score>=2],['high','medium'],default='low')
review['proposed_action']=np.select([
    review.context_risk>=1,
    review.quality_repetition_flag.eq(1),
    review.quality_very_short_flag.eq(1),
    review.quality_very_long_flag.eq(1)],
    ['VERIFY_CONTEXT_RECONSTRUCT_OR_EXCLUDE','REVIEW_REDUNDANCY','REVIEW_COMPLETENESS','REVIEW_CONCISION'],default='MANUAL_FACTUAL_REVIEW')
review['curation_decision']='PENDING_HUMAN_REVIEW'
review['decision_evidence']=''
review['verified_source']=''
review['reviewer']=''
review['review_date']=''
review=review.sort_values(['priority_score','review_id'],ascending=[False,True])
print(review.groupby(['priority','language']).size().unstack(fill_value=0).to_string())


## 7. Exportações auditáveis e relatório

A planilha de revisão inclui campos para evidência, decisão e fonte verificada. Nenhuma decisão é preenchida automaticamente.

In [ ]:
review.to_csv(OUTPUT_DIR/'contextual_review_queue.csv',index=False,encoding='utf-8-sig')
if not links_df.empty:links_df.to_csv(OUTPUT_DIR/'benchmark_corpus_exploratory_links.csv',index=False,encoding='utf-8-sig')
else:pd.DataFrame(columns=['review_id','corpus_id','language','benchmark_question_id','link_status']).to_csv(OUTPUT_DIR/'benchmark_corpus_exploratory_links.csv',index=False)

summary={
 'audit':'maia_sft_contextual_v2','status':'AUTOMATED_TRIAGE_COMPLETE_HUMAN_FACTUAL_REVIEW_PENDING',
 'sample_size':len(review),'source_file':str(review_path),
 'benchmark_file':str(benchmark_path) if benchmark_path else None,
 'counts_by_priority':{str(k):int(v) for k,v in review.priority.value_counts().items()},
 'counts_by_language':{str(k):int(v) for k,v in review.language.value_counts().items()},
 'context_flagged':int((review.context_risk>0).sum()),
 'repetition_flagged':int(review.quality_repetition_flag.sum()),
 'benchmark_exploratory_links':len(links_df),
 'caveats':['Sample is risk-enriched; percentages cannot be extrapolated to full corpus.',
            'Heuristics do not establish factual errors.',
            'Lexical benchmark links do not establish training causality.',
            'No corpus records were modified or removed.']}
(OUTPUT_DIR/'contextual_audit_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')

report=f"""# Maia Lite — Auditoria contextual v2

- Amostra analisada: **{len(review)}** exemplos (amostra enriquecida por risco; não representativa do corpus completo).
- Suspeitas de contexto ausente: **{summary['context_flagged']}**.
- Sinais de repetição: **{summary['repetition_flagged']}**.
- Ligações lexicais exploratórias com benchmark: **{len(links_df)}**.
- Prioridade alta: **{summary['counts_by_priority'].get('high',0)}**; média: **{summary['counts_by_priority'].get('medium',0)}**; baixa: **{summary['counts_by_priority'].get('low',0)}**.

## Interpretação

A triagem não confirma erros factuais, não estima a prevalência de problemas no corpus completo e não estabelece causalidade entre exemplos e gerações do benchmark. Revisar a fila e registrar evidência antes de corrigir ou excluir dados.

## Decisão

**Não treinar v2.4 automaticamente.** Realizar revisão humana dos itens prioritários, documentar fontes, e só então gerar uma proposta de patch versionado do corpus.
"""
(OUTPUT_DIR/'contextual_audit_report.md').write_text(report,encoding='utf-8')
print(report)
print('Arquivos:',*[p.name for p in OUTPUT_DIR.iterdir()],sep='\n - ')


## 8. Protocolo de revisão humana

Preencha `contextual_review_queue.csv` sem modificar os prompts originais:

- `curation_decision`: `KEEP`, `REWRITE_WITH_SOURCE`, `RESTORE_CONTEXT`, `EXCLUDE`, ou `NEEDS_EXPERT`.
- `decision_evidence`: justificativa específica, citando trecho do prompt/resposta.
- `verified_source`: referência científica ou contexto conversacional recuperado (quando aplicável).
- `reviewer`, `review_date`: responsável e data.

**Critério de promoção:** revisar casos de alta prioridade, medir taxas por idioma/fonte em amostra aleatória separada, validar respostas científicas com evidência, e repetir o benchmark controlado. O notebook não aplica patches.